In [1]:
#Importing 
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import joblib
import pickle

In [6]:
# ── Load processed data ───────────────────────────────────────
df=pd.read_csv("../data/processed/processed_reviews.csv")
syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

# ── Load TF-IDF ───────────────────────────────────────────────
tfidf     = joblib.load("../models/traditional/tfidf.pkl")

# ── Load feature list ─────────────────────────────────────────
with open("../models/meta_features.pkl", "rb") as f:
    all_meta = pickle.load(f)

print("✅ All artifacts loaded — ready to train!")

✅ All artifacts loaded — ready to train!


In [ ]:
# ── Combine TF-IDF + metadata features ───────────────────────
from scipy.sparse import csr_matrix

def get_features(df, tfidf, fit=False):
    if fit:
        text_feats = tfidf.fit_transform(df["review_text"])
    else:
        text_feats = tfidf.transform(df["review_text"])
    meta_feats = csr_matrix(df[all_meta].values)
    return hstack([text_feats, meta_feats])


In [8]:
X_train = get_features(syn_train, tfidf, fit=True)
X_val   = get_features(syn_val,   tfidf)
X_test  = get_features(syn_test,  tfidf)
y_train = syn_train["label"]
y_val   = syn_val["label"]
y_test  = syn_test["label"]

In [9]:
# ── Train each model ──────────────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, C=1.0, class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10,
        class_weight="balanced", random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6,
        learning_rate=0.1, use_label_encoder=False,
        eval_metric="logloss", random_state=42
    ),
    "LinearSVC": LinearSVC(
        C=1.0, class_weight="balanced", max_iter=2000
    ),
}

results = {}
for name, model in models.items():
    print(f"\n{'='*45}")
    print(f"  Training: {name}")
    print(f"{'='*45}")

    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)

    print(classification_report(y_val, val_preds,
          target_names=["genuine", "fake"]))

    results[name] = {
        "model": model,
        "val_preds": val_preds
    }
    joblib.dump(model, f"{name.replace(' ','_')}.pkl")


  Training: Logistic Regression


c:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

     genuine       0.90      0.95      0.93       218
        fake       0.95      0.90      0.93       229

    accuracy                           0.93       447
   macro avg       0.93      0.93      0.93       447
weighted avg       0.93      0.93      0.93       447


  Training: Random Forest
              precision    recall  f1-score   support

     genuine       0.91      0.87      0.89       218
        fake       0.88      0.92      0.90       229

    accuracy                           0.89       447
   macro avg       0.89      0.89      0.89       447
weighted avg       0.89      0.89      0.89       447


  Training: XGBoost


c:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [02:31:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

     genuine       0.93      0.97      0.95       218
        fake       0.97      0.93      0.95       229

    accuracy                           0.95       447
   macro avg       0.95      0.95      0.95       447
weighted avg       0.95      0.95      0.95       447


  Training: LinearSVC
              precision    recall  f1-score   support

     genuine       0.95      0.66      0.78       218
        fake       0.75      0.97      0.84       229

    accuracy                           0.82       447
   macro avg       0.85      0.81      0.81       447
weighted avg       0.85      0.82      0.81       447



c:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
